## Variational GPR on WeatherBench Data 
Apply GP regression baseline


In [1]:
# Check GPU
!nvidia-smi

Sun Feb 23 16:40:13 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.12             Driver Version: 535.104.12   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100 80GB PCIe          Off | 00000000:1B:00.0 Off |                    0 |
| N/A   43C    P0              61W / 300W |      4MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import pickle
import time
from tqdm.notebook import tqdm

import torch
from torch.utils.data import TensorDataset, DataLoader

import gpytorch
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution, VariationalStrategy

import geometric_kernels
import geometric_kernels.torch 
from geometric_kernels.spaces import Hypersphere
from geometric_kernels.kernels import MaternGeometricKernel
from geometric_kernels.frontends.gpytorch import GPyTorchGeometricKernel

INFO (geometric_kernels): Numpy backend is enabled. To enable other backends, don't forget to `import geometric_kernels.*backend name*`.
INFO (geometric_kernels): We may be suppressing some logging of external libraries. To override the logging policy, call `logging.basicConfig`.
INFO (geometric_kernels): Torch backend enabled.


In [3]:
torch.set_default_dtype(torch.float64)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


A problem with GeometricKernel computations leading to tensors not on same device (check geometric_kernels.kernels.karhunen_loeve line 141)

In [5]:
import geometric_kernels.kernels.karhunen_loeve as kl
from geometric_kernels.spaces.eigenfunctions import Eigenfunctions
import geometric_kernels.spaces.hypersphere as spaces
import lab as B
from geometric_kernels.lab_extras import complex_like, is_complex

def new_spectrum(s, nu, lengthscale, dimension):

    assert lengthscale.shape == (1,)
    assert nu.shape == (1,)

    s_tensor = torch.as_tensor(s, dtype=lengthscale.dtype, device=lengthscale.device)
    # Replace np.r_[1.0] with a torch tensor on the correct device
    one_tensor = torch.tensor([1.0], dtype=lengthscale.dtype, device=lengthscale.device)
    # Compute safe_nu: if nu == inf, use 1.0; otherwise keep nu
    safe_nu = torch.where(nu == np.inf, one_tensor, nu)
    # For nu == inf: compute spectral values
    spectral_values_nu_infinite = torch.exp(- (lengthscale**2) / 2.0 * s_tensor).to(device)
    
    # For nu < inf: compute spectral values
    power = -safe_nu - dimension / 2.0
    base = 2.0 * safe_nu / (lengthscale**2) + s_tensor
    spectral_values_nu_finite = base ** power
    
    return torch.where(nu == np.inf, spectral_values_nu_infinite, spectral_values_nu_finite)


def patched_addition_theorem(self, X, X2=None, **kwargs):
    # Determine target device from input X (assumes X is a tensor)
    device = X.device if hasattr(X, "device") else torch.device("cuda")
    # Compute the values and force them to the target device
    values = [
        level.addition(X, X2)[..., None].to(device)  # [N, N2, 1]
        for level in self._spherical_harmonics.harmonic_levels
    ]
    # Concatenate along the last dimension; using torch.cat here
    return torch.cat(values, dim=-1)  # [N, N2, L]

def patched_weighted_outerproduct_diag(self, weights, X, **kwargs):
    # Determine target device from input X (assumes X is a tensor)
    device = X.device if hasattr(X, "device") else torch.device("cuda")

    phi_product_diag = self.phi_product_diag(X, **kwargs)  # [N, L]

    if is_complex(phi_product_diag):
        phi_product_diag = B.cast(complex_like(weights), phi_product_diag)
        weights = B.cast(complex_like(weights), weights)
    else:
        phi_product_diag = B.cast(B.dtype(weights), phi_product_diag)

    return B.einsum("id,ni->n", weights.to(device), phi_product_diag.to(device)) 

# Apply the monkey patch
kl.MaternKarhunenLoeveKernel.spectrum = staticmethod(new_spectrum)
spaces.SphericalHarmonics._addition_theorem = patched_addition_theorem
Eigenfunctions.weighted_outerproduct_diag = patched_weighted_outerproduct_diag

In [6]:
def to_pickle(obj, fn):
    with open(fn, 'wb') as f:
        pickle.dump(obj, f)
def read_pickle(fn):
    with open(fn, 'rb') as f:
        return pickle.load(f)

In [7]:
def compute_weighted_mae(da_fc, da_true, mean_dims=xr.ALL_DIMS):
    """ Stolen from WeatherBench score.py
    Compute the MAE with latitude weighting from two xr.DataArrays.
    Args:
        da_fc (xr.DataArray): Forecast. Time coordinate must be validation time.
        da_true (xr.DataArray): Truth.
        mean_dims: dimensions over which to average score
    Returns:
        mae: Latitude weighted root mean absolute error
    """
    error = da_fc - da_true
    weights_lat = np.cos(np.deg2rad(error.lat))
    weights_lat /= weights_lat.mean()
    mae = (np.abs(error) * weights_lat).mean(mean_dims)
    return mae

def compute_weighted_rmse(da_fc, da_true, mean_dims=xr.ALL_DIMS):
    """ Stolen from WeatherBench score.py
    Compute the RMSE with latitude weighting from two xr.DataArrays.

    Args:
        da_fc (xr.DataArray): Forecast. Time coordinate must be validation time.
        da_true (xr.DataArray): Truth.
        mean_dims: dimensions over which to average score
    Returns:
        rmse: Latitude weighted root mean squared error
    """
    error = da_fc - da_true
    weights_lat = np.cos(np.deg2rad(error.lat))
    weights_lat /= weights_lat.mean()
    rmse = np.sqrt(((error)**2 * weights_lat).mean(mean_dims))
    return rmse

## Data Processing

Loading the data

In [8]:
z500 = xr.open_mfdataset('STGP-weather-forecasting/data/5.625deg/geopotential_500/*.nc', combine='by_coords')
z500_train = z500.sel(time=slice('2017.12', '2017.12'))['z']
z500_test = z500.sel(time=slice('2018.1', '2018.1.7'))['z'] # 2017-2018 data

In [9]:
data_mean = z500_train.mean().load()
data_std = z500_train.std().load()

# Normalize datasets
data_train = (z500_train - data_mean) / data_std
data_test = (z500_test - data_mean) / data_std

We need to convert (lat., lon.) coordinates into (x,y,z) for the inputs into the spatial kernel from GeometricKernel 

In [10]:
def latlon_to_cartesian(lat, lon, device=device):
    """ Converting (lat., lon.) to cartesian coordinates on a unit sphere
        Note: WeatherBench data is defined on a constant altitude """

    lat, lon = np.deg2rad(lat), np.deg2rad(lon)

    x = np.cos(lat)[:, None] * np.cos(lon)[None, :]
    y = np.cos(lat)[:, None] * np.sin(lon)[None, :]
    z = np.sin(lat)[:, None] * np.ones_like(lon)[None, :]

    xyz = np.stack([x, y, z], axis=-1)  # Out: (nlat, nlon, 3)
    return torch.tensor(xyz).to(device)

Subsampling the datasets due to memory issues

In [11]:
def create_data(data_train, data_test, lead_time_h, space_subsample=1, time_subsample=1, device=device, train=True):
    """Preparing input and output data. X should be the coordinate input into the kernel, 
    while Y should be the forecast target (shifted lead time)."""

    # Training Data
    X = data_train.isel(time=slice(0, -lead_time_h, time_subsample),
                        lat=slice(0, None, space_subsample),
                        lon=slice(0, None, space_subsample))
    Y = data_train.isel(time=slice(lead_time_h, None, time_subsample),
                  lat=slice(0, None, space_subsample),
                  lon=slice(0, None, space_subsample)) 

    # Preparing the input coords (should have shape (ntime, nlat, nlon, 4) if we have no subsampling)
    X_time = (X.time - data_train.time[0]).values / np.timedelta64(1, 'h')
    X_time = torch.tensor(X_time).to(device)
    t_steps = X_time.shape[0]

    latitudes = X.lat.values 
    longitudes = X.lon.values 
    cartesian_coords = latlon_to_cartesian(latitudes, longitudes, device)
    nlat, nlon, _ = cartesian_coords.shape

    if train:
        print("Processing Training Dataset")
        # Combine spatial and temporal data
        training_coords = torch.empty((t_steps, nlat, nlon, 4)).to(device)
        # Fill in the spatial part first
        training_coords[..., :3] = cartesian_coords.unsqueeze(0).expand(t_steps, -1, -1, -1)
        # The final element is time
        training_coords[..., 3] = X_time.view(t_steps, 1, 1).expand(t_steps, nlat, nlon)
        X_size = training_coords.element_size() * training_coords.nelement() / (1024**2)
        print(f"Combined coords info --> Shape: {training_coords.shape}, Mem: {X_size:.2f} MB")

        # Flatten to shape (N,)
        Y = torch.tensor(Y.values).view(-1).to(device)
        Y_size = Y.element_size() * Y.nelement() / (1024**2)
        print(f"Target info --> Shape: {Y.shape}, Mem: {Y_size:.2f} MB")

        return training_coords, Y
    
    else:
        # Testing Data
        X_test = data_test.isel(time=slice(0, -lead_time_h, time_subsample),
                            lat=slice(0, None, space_subsample),
                            lon=slice(0, None, space_subsample))
        Y_test = data_test.isel(time=slice(lead_time_h, None, time_subsample),
                    lat=slice(0, None, space_subsample),
                    lon=slice(0, None, space_subsample))
        X_test_time = (X_test.time - data_train.time[0]).values / np.timedelta64(1, 'h')
        X_test_time = torch.tensor(X_test_time).to(device)
        test_t_steps = X_test_time.shape[0]

        print("Processing Test Dataset")
        # Combine spatial and temporal data
        testing_coords = torch.empty((test_t_steps, nlat, nlon, 4)).to(device)
        # Fill in the spatial part first
        testing_coords[..., :3] = cartesian_coords.unsqueeze(0).expand(test_t_steps, -1, -1, -1)
        # The final element is time
        testing_coords[..., 3] = X_test_time.view(test_t_steps, 1, 1).expand(test_t_steps, nlat, nlon)
        X_size = testing_coords.element_size() * testing_coords.nelement() / (1024**2)
        print(f"Combined coords info --> Shape: {testing_coords.shape}, Mem: {X_size:.2f} MB")

        # Flatten to shape (N,)
        Y_test = torch.tensor(Y_test.values).view(-1).to(device)
        Y_size = Y_test.element_size() * Y_test.nelement() / (1024**2)
        print(f"Target info --> Shape: {Y_test.shape}, Mem: {Y_size:.2f} MB")

        return testing_coords, Y_test, latitudes, longitudes

## Defining the Kernel

We work on the 2d-hypersphere, modeling the spatial kernel with Geometric_Kernel and time kernel with periodic kernel. 

In [ ]:
# Spatial Kernel from Geometric_Kernel package
sphere = Hypersphere(dim=2)
# Matern Kernel
spatial_kernel_gm = MaternGeometricKernel(sphere)
params = spatial_kernel_gm.init_params()
# Set params here, we want to work with Matern-3/2 (or 5/2)
params["lengthscale"] = torch.tensor([0.5]).to(device) #TODO: try 3
params["nu"] = torch.tensor([3/2]).to(device)
print('params:', params)

params: {'nu': tensor([1.5000], device='cuda:0'), 'lengthscale': tensor([0.5000], device='cuda:0')}


In [ ]:
# Define the GP regression kernel, separable in space and time
# First convert Geometric Kernel to a GPyTorch compatible kernel
#spatial_kernel = gpytorch.kernels.ScaleKernel(
#                    GPyTorchGeometricKernel(
#                        spatial_kernel_gm,
#                        nu = params["nu"],
#                        lengthscale=params["lengthscale"],
#                        trainable_nu=False,
#                        active_dims=[0,1,2]
#                    )
#                ).to(device)
# spatial_kernel.outputscale = 1.0 #Fix the scale of cov function

matern_spatial = gpytorch.kernels.MaternKernel(nu=1.5, active_dims=[0,1,2]).to(device)
matern_spatial.lengthscale = torch.tensor(0.5).to(device)
spatial_kernel = gpytorch.kernels.ScaleKernel(matern_spatial).to(device)

# For temporal kernel
time_kernel1 = gpytorch.kernels.PeriodicKernel(period_length=24, active_dims=[3]).to(device) # Daily   
time_kernel2 = gpytorch.kernels.PeriodicKernel(period_length=24*3, active_dims=[3]).to(device) # 3-Days
time_kernel3 = gpytorch.kernels.PeriodicKernel(period_length=24*7, active_dims=[3]).to(device) # Weekly

time_kernel1.register_prior("period_length_prior", gpytorch.priors.NormalPrior(24*7, 3), "period_length")
time_kernel2.register_prior("period_length_prior", gpytorch.priors.NormalPrior(24*30, 9), "period_length")
time_kernel3.register_prior("period_length_prior", gpytorch.priors.NormalPrior(24*7, 21), "period_length")

time_kernel = time_kernel1 + time_kernel2 + time_kernel3

# Construct the product kernel
kernel = spatial_kernel * time_kernel

# Approximate GP
class GPModel(ApproximateGP):
    def __init__(self, inducing_points):
        # inducing_points: tensor of shape (M, 4)
        variational_distribution = CholeskyVariationalDistribution(inducing_points.size(0))
        variational_strategy = VariationalStrategy(self, inducing_points, variational_distribution, learn_inducing_locations=True)
        super(GPModel, self).__init__(variational_strategy)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = kernel

    def forward(self, x):
        # x is expected to be of shape (N, 4) (flattened combined coords)
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

## Training and Evaluation

In [13]:
def train_gp(lead_time, data_train, data_test, space_subsample_ips, time_subsample_ips, space_subsample=1, time_subsample=1, epochs=5, batch_size=2048, save_model=False, device=device):
    """
    Train a Gaussian Process model using GPyTorch. Can add subsampling the data for efficiency.
    """
    
    # Prepare training data (Here we do not subsample any of the training data)
    X_train, Y_train = create_data(data_train, data_test, lead_time, space_subsample, time_subsample)

    # Inducing points technique
    inducing_points = X_train[::time_subsample, ::space_subsample, ::space_subsample,:].reshape(-1, 4)
    X_train = X_train.view(-1,4) # Shape: (ntime*nlat*nlon, 4)
 
    train_dataset = TensorDataset(X_train, Y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
    model = GPModel(inducing_points).to(device)
    
    # Train model
    model.train()
    likelihood.train()

    # Optimizer
    optimizer = torch.optim.Adam([
        {'params': model.parameters()},
        {'params': likelihood.parameters()},
    ], lr=0.05)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.99)

    # Loss fn
    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=len(train_dataset))

    print("Starting Variational GP regression training:")
    # Reset the peak memory statistics before training
    torch.cuda.reset_peak_memory_stats(device)
    loss_history = []
    start_time = time.time()

    for epoch in range(epochs):
        epoch_loss = 0.0
        batch_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(x_batch)
            # Scale the loss to approximate full-data loss
            loss = -mll(output, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            batch_bar.set_postfix(loss=f"{loss.item():.3f}")
            batch_bar.update(1)
            loss_history.append(loss.item())
        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss/len(train_loader):.3f}")
            
    current_allocated = torch.cuda.memory_allocated(device)
    max_allocated = torch.cuda.max_memory_allocated(device)
    print(f"Current GPU memory allocated: {current_allocated / (1024**2):.2f} MB")
    print(f"Peak GPU memory allocated: {max_allocated / (1024**2):.2f} MB")

    end_time = time.time()
    print("Training completed")
    time_elapsed = end_time - start_time
    print(f"Training time: {time_elapsed:.2f}s")

    # Save model
    if save_model:
        torch.save(model.state_dict(), "STGP-weather-forecasting/saved_models/vargp_model.pth")
        torch.save(likelihood.state_dict(), "STGP-weather-forecasting/saved_models/vargp_likelihood.pth")
        print("Saved model")

    return model, likelihood, loss_history

In [14]:
lead_time = 24*3 # units of hours
space_subsample_ips = 5 # regular subsampling as inducing points
time_subsample_ips = 8 # regular subsampling as inducing points
space_subsample = 1
time_subsample = 6

# Train GP model
model, likelihood, loss = train_gp(lead_time, data_train, data_test, space_subsample_ips, time_subsample_ips, space_subsample, time_subsample)

Processing Training Dataset
Combined coords info --> Shape: torch.Size([112, 32, 64, 4]), Mem: 7.00 MB
Target info --> Shape: torch.Size([229376]), Mem: 0.88 MB
Starting Variational GP regression training:


Epoch 1/5:   0%|          | 0/112 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 608.00 MiB. GPU 0 has a total capacity of 79.15 GiB of which 257.25 MiB is free. Including non-PyTorch memory, this process has 78.89 GiB memory in use. Of the allocated memory 78.40 GiB is allocated by PyTorch, and 2.15 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Plot the loss
plt.plot(loss, label='Training Loss (MLL)')
plt.xlabel("Epoch")
plt.ylabel("Negative Marginal Log Likelihood")
plt.title("Training Loss Over Epochs")
plt.legend()
plt.show()

In [ ]:
X_test, Y_test, lat, lon = create_data(data_train, data_test, lead_time, space_subsample=space_subsample, 
                                                time_subsample=time_subsample, device=device, train=False)

test_dataset = TensorDataset(X_test, Y_test)
test_loader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

nlat = len(lat)
nlon = len(lon)

# Make Predictions
model.eval()
likelihood.eval()
preds = []
with torch.no_grad():
    for x_batch, _ in test_loader:
        batch_pred = likelihood(model(x_batch))
        preds.append(batch_pred.mean.cpu())
y_pred_flat = torch.cat(preds, dim=0)  

N_test = y_pred_flat.size(0)
T_test = N_test // (nlat * nlon)
y_pred_mean = y_pred_flat.view(T_test, nlat, nlon).numpy()

test_times = z500_test.time.values[::time_subsample][:T_test]

y_pred_xr = xr.DataArray(y_pred_mean, dims=['time', 'lat', 'lon'], 
                         coords={'time': test_times, 'lat': lat, 'lon': lon})
rmse = compute_weighted_rmse(y_pred_xr, z500_test)
print(f"Test RMSE: {rmse.values:.3f}")

Test RMSE: 1524.214
